In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/12 12:31:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=pyspark-shell>

# **Tabular Data**

In [3]:
my_grocery_list = [
    ["Banana", 2, 1.74],
    ["Apple", 4, 2.04],
    ["Carrot", 1, 1.09],
    ["Cake", 1, 10.99],
]

df_grocery_list = spark.createDataFrame(
    my_grocery_list, ["Item", "Quantity", "Price"]
)

df_grocery_list.printSchema()

root
 |-- Item: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- Price: double (nullable = true)



![](../images/Cluster_Memory.png)

In [4]:
import os

In [5]:
DIRECTORY = "../data/broadcast_logs"
df_logs = spark.read.csv(
    os.path.join(DIRECTORY, 'BroadcastLogs_2018_Q3_M8_sample.CSV'),
    sep='|',
    inferSchema=True,
    header=True,
    timestampFormat="yyyy-MM-dd"
)

df_logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

# **Data Manipulation**

## **Selecting columns**

In [6]:
df_logs.select("BroadcastLogID", "LogServiceID", "LogDate").show(5, False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows


In [7]:
df_logs.select(*["BroadcastLogID", "LogServiceID", "LogDate"]).show(5, False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows


In [8]:
import numpy as np

In [9]:
splited_columns = np.array_split(
    df_logs.columns, len(df_logs.columns) // 3
)


for x in splited_columns:
    df_logs.select(*x).show(5, truncate=False)

+--------------+------------+----------+
|BroadcastLogID|LogServiceID|LogDate   |
+--------------+------------+----------+
|1196192316    |3157        |2018-08-01|
|1196192317    |3157        |2018-08-01|
|1196192318    |3157        |2018-08-01|
|1196192319    |3157        |2018-08-01|
|1196192320    |3157        |2018-08-01|
+--------------+------------+----------+
only showing top 5 rows
+----------+-------------------+----------------------+
|SequenceNO|AudienceTargetAgeID|AudienceTargetEthnicID|
+----------+-------------------+----------------------+
|1         |4                  |NULL                  |
|2         |NULL               |NULL                  |
|3         |NULL               |NULL                  |
|4         |NULL               |NULL                  |
|5         |NULL               |NULL                  |
+----------+-------------------+----------------------+
only showing top 5 rows
+----------+---------------+-----------------+
|CategoryID|ClosedCaptionID|Coun

## **Deleting columns**

In [10]:
df_logs = df_logs.drop("BroadcastLogID", "SequenceNO")

print("BroadcastLogID" in df_logs.columns) # => False
print("SequenceNo" in df_logs.columns) # => False

False
False


**Advanced topic: An unfortunate inconsistency**

In theory, you can also select() columns with a list without unpacking them. This code will work as expected:
```spark
logs = logs.select(
    [x for x in logs.columns if x not in ["BroadcastLogID", "SequenceNO"]]
)
```
This is not the case for drop(), where you need to explicitly unpack:
```spark
logs.drop(logs.columns[:])
# TypeError: col should be a string or a Column
logs.drop(*logs.columns[:])
# DataFrame[]
```

## **New columns with `withColumn()`**

In [11]:
df_logs.select(F.col("Duration")).show(5)

+----------------+
|        Duration|
+----------------+
|02:00:00.0000000|
|00:00:30.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
|00:00:15.0000000|
+----------------+
only showing top 5 rows


In [12]:
df_logs.select(F.col('Duration')).dtypes

[('Duration', 'string')]

In [13]:
df_logs.select(
    F.col("Duration"),
    F.col("Duration").substr(1, 2).cast("int").alias("dur_hours"),
    F.col("Duration").substr(4, 2).cast("int").alias("dur_minutes"),
    F.col("Duration").substr(7, 2).cast("int").alias("dur_seconds"),
).distinct().show(5, False)

+----------------+---------+-----------+-----------+
|Duration        |dur_hours|dur_minutes|dur_seconds|
+----------------+---------+-----------+-----------+
|00:04:52.0000000|0        |4          |52         |
|00:10:06.0000000|0        |10         |6          |
|00:26:41.0000000|0        |26         |41         |
|00:05:29.0000000|0        |5          |29         |
|00:08:18.0000000|0        |8          |18         |
+----------------+---------+-----------+-----------+
only showing top 5 rows


In [14]:
df_logs.select(
    F.col("Duration"),
    (
        F.col("Duration").substr(1, 2).cast('int') * 120 
        + F.col("Duration").substr(4, 2).cast('int') * 60 
        + F.col("Duration").substr(7, 2).cast('int') 
    ).alias("Duration in second")
).distinct().show(5, False)

+----------------+------------------+
|Duration        |Duration in second|
+----------------+------------------+
|00:31:00.0000000|1860              |
|00:28:08.0000000|1688              |
|00:10:30.0000000|630               |
|00:32:00.0000000|1920              |
|00:30:00.0000000|1800              |
+----------------+------------------+
only showing top 5 rows


In [15]:
df_logs = df_logs.withColumn(
    "Duration_Seconds",
    (
        F.col("Duration").substr(1, 2).cast('int') * 120 
        + F.col("Duration").substr(4, 2).cast('int') * 60 
        + F.col("Duration").substr(7, 2).cast('int') 
    )
)

df_logs.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

> **WARNING :** If you create a column withColumn() and give it a name that already exists in your data frame, PySpark will happily overwrite the column.

> **WARNING :** Creating many (100+) new columns using `withColumns()` will slow Spark down to a grind. If you need to create a lot of columns at once, use the `select()` approach. While it will generate the same work, it is less tasking on the query planner.

![](../images/select_versus_withColumn.png)

## **Renaming and reordering columns**

In [16]:
df_logs = df_logs.withColumnRenamed(
    'Duration_Seconds', 'duration_seconds'
)

df_logs.printSchema()

root
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

In [17]:
df_logs.toDF(*[x.lower() for x in df_logs.columns]).printSchema()

root
 |-- logserviceid: integer (nullable = true)
 |-- logdate: date (nullable = true)
 |-- audiencetargetageid: integer (nullable = true)
 |-- audiencetargetethnicid: integer (nullable = true)
 |-- categoryid: integer (nullable = true)
 |-- closedcaptionid: integer (nullable = true)
 |-- countryoforiginid: integer (nullable = true)
 |-- dubdramacreditid: integer (nullable = true)
 |-- ethnicprogramid: integer (nullable = true)
 |-- productionsourceid: integer (nullable = true)
 |-- programclassid: integer (nullable = true)
 |-- filmclassificationid: integer (nullable = true)
 |-- exhibitionid: integer (nullable = true)
 |-- duration: string (nullable = true)
 |-- endtime: string (nullable = true)
 |-- logentrydate: date (nullable = true)
 |-- productionno: string (nullable = true)
 |-- programtitle: string (nullable = true)
 |-- starttime: string (nullable = true)
 |-- subtitle: string (nullable = true)
 |-- networkaffiliationid: integer (nullable = true)
 |-- specialattentionid: inte

In [18]:
df_logs.select(sorted(df_logs.columns)).printSchema()

root
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- BroadcastOriginPointID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CompositionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- Language1: integer (nullable = true)
 |-- Language2: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- Producer1: string (nullable = true)
 |-- Producer2: string (nullable = true)
 |-- ProductionNO: string (nu

## **Diagnosing a Data Frame**

In [19]:
for x in df_logs.columns:
    print(df_logs[x].dtype)
    df_logs.describe(x).show()

Column<'LogServiceID['dtype']'>
+-------+------------------+
|summary|      LogServiceID|
+-------+------------------+
|  count|            238945|
|   mean| 3450.890284375065|
| stddev|199.50673962555987|
|    min|              3157|
|    max|              3925|
+-------+------------------+

Column<'LogDate['dtype']'>
+-------+
|summary|
+-------+
|  count|
|   mean|
| stddev|
|    min|
|    max|
+-------+

Column<'AudienceTargetAgeID['dtype']'>
+-------+-------------------+
|summary|AudienceTargetAgeID|
+-------+-------------------+
|  count|              16112|
|   mean| 3.4929245283018866|
| stddev|  1.041596339474513|
|    min|                  1|
|    max|                  4|
+-------+-------------------+

Column<'AudienceTargetEthnicID['dtype']'>
+-------+----------------------+
|summary|AudienceTargetEthnicID|
+-------+----------------------+
|  count|                  1710|
|   mean|    120.56432748538012|
| stddev|     71.98694059436134|
|    min|                     4|
|    

In [20]:
for x in df_logs.columns:
    df_logs.select(x).summary("min", "10%", "90%", "max").show()

+-------+------------+
|summary|LogServiceID|
+-------+------------+
|    min|        3157|
|    10%|        3236|
|    90%|        3709|
|    max|        3925|
+-------+------------+

+-------+
|summary|
+-------+
|    min|
|    10%|
|    90%|
|    max|
+-------+

+-------+-------------------+
|summary|AudienceTargetAgeID|
+-------+-------------------+
|    min|                  1|
|    10%|                  1|
|    90%|                  4|
|    max|                  4|
+-------+-------------------+

+-------+----------------------+
|summary|AudienceTargetEthnicID|
+-------+----------------------+
|    min|                     4|
|    10%|                    74|
|    90%|                   258|
|    max|                   337|
+-------+----------------------+

+-------+----------+
|summary|CategoryID|
+-------+----------+
|    min|         1|
|    10%|         3|
|    90%|        29|
|    max|        29|
+-------+----------+

+-------+---------------+
|summary|ClosedCaptionID|
+------

+-------+--------------+
|summary|ProgramClassID|
+-------+--------------+
|    min|             3|
|    10%|             3|
|    90%|            20|
|    max|            30|
+-------+--------------+

+-------+--------------------+
|summary|FilmClassificationID|
+-------+--------------------+
|    min|                   1|
|    10%|                   1|
|    90%|                   5|
|    max|                   8|
+-------+--------------------+

+-------+------------+
|summary|ExhibitionID|
+-------+------------+
|    min|           1|
|    10%|           2|
|    90%|           5|
|    max|           8|
+-------+------------+

+-------+----------------+
|summary|        Duration|
+-------+----------------+
|    min|00:00:01.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|06:30:09.0000000|
+-------+----------------+



+-------+----------------+
|summary|         EndTime|
+-------+----------------+
|    min|00:00:00.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|23:59:59.0000000|
+-------+----------------+

+-------+
|summary|
+-------+
|    min|
|    10%|
|    90%|
|    max|
+-------+

+-------+------------+
|summary|ProductionNO|
+-------+------------+
|    min|      030641|
|    10%|     30641.0|
|    90%|     39949.0|
|    max|      c34183|
+-------+------------+

+-------+------------------+
|summary|      ProgramTitle|
+-------+------------------+
|    min|  !NO 5PM A ID(:5)|
|    10%|            1999.0|
|    90%|            1999.0|
|    max|�t� avec Jo�l 2/Un|
+-------+------------------+

+-------+----------------+
|summary|       StartTime|
+-------+----------------+
|    min|00:00:00.0000000|
|    10%|            NULL|
|    90%|            NULL|
|    max|23:59:59.0000000|
+-------+----------------+

+-------+--------------------+
|summary|            Subtitle|
+----

# **Additional exercises**

#### **Exercise 4.3**
Reread the data in a logs_raw data frame (the data file is `./data/broadcast_logs-BroadcastLogs_2018_Q3_M8.CSV)`, this time without passing any optional parameters Print the first five rows of data, as well as the schema. What are the differences in terms of data and schema between logs and logs_raw?

In [21]:
df_logs_raw = spark.read.csv('../data/broadcast_logs/BroadcastLogs_2018_Q3_M8_sample.CSV')

df_logs_raw.printSchema()

root
 |-- _c0: string (nullable = true)



In [22]:
df_logs_raw.show(5, False)

+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|_c0                                                                                                                                                                                                                                                                                                                                                                                                                                        |
+-------------------------------------------------------------------------------------------------------------------------------------------

#### **Exercise 4.4**
Create a new data frame, logs_clean, that contains only the columns that do not end with ID.

In [23]:
DIRECTORY

'../data/broadcast_logs'

In [25]:
df_clean_logs = spark.read.csv(
    os.path.join(DIRECTORY, 'BroadcastLogs_2018_Q3_M8_sample.CSV'),
    sep= '|',
    inferSchema=True,
    header=True,
    timestampFormat='yyyy/MM/DD'
)

df_clean_logs.printSchema()

root
 |-- BroadcastLogID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- ProgramClassID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable 

In [26]:
selected_cols = [x for x in df_clean_logs.columns if x.endswith("ID")]

df_clean_logs.drop(*selected_cols).printSchema()

root
 |-- LogDate: date (nullable = true)
 |-- SequenceNO: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- Producer1: string (nullable = true)
 |-- Producer2: string (nullable = true)
 |-- Language1: integer (nullable = true)
 |-- Language2: integer (nullable = true)

